In [1]:
import sys, os
import pandas as pd
import numpy as np
import json
from pathlib import Path
# Para SMOTE (balanceo)


# Add project root to sys.path
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.data_enrichment import get_features

# Correct path to your data/raw folder
RAW_DIR = "../data/raw"

df_feats, feature_cols = get_features(RAW_DIR)

In [6]:
# Build ML dataset
df_ml = df_feats[
    (df_feats["season_end_year"] >= 2008) &
    (df_feats["season_end_year"] <= 2024)
].copy()
print(df_ml.shape)



(51077, 76)


In [3]:
# Temporal split for ML training
df_train = df_ml[df_ml["season_end_year"] <= 2018].copy()
df_val   = df_ml[(df_ml["season_end_year"] >= 2019) &
                 (df_ml["season_end_year"] <= 2022)].copy()
df_test  = df_ml[df_ml["season_end_year"] >= 2023].copy()

print("Train:", df_train.shape)
print("Val:  ", df_val.shape)
print("Test: ", df_test.shape)
print("Total:", df_train.shape[0] + df_val.shape[0] + df_test.shape[0])


Train: (30530, 76)
Val:   (13482, 76)
Test:  (7065, 76)
Total: 51077


In [4]:
# Feature matrix and target for each split
X_train = df_train[feature_cols].copy()
y_train = df_train["ballon_dor_winner"].astype(int)

X_val = df_val[feature_cols].copy()
y_val = df_val["ballon_dor_winner"].astype(int)

X_test = df_test[feature_cols].copy()
y_test = df_test["ballon_dor_winner"].astype(int)

print("Train X:", X_train.shape, "Train y:", y_train.shape)
print("Val   X:", X_val.shape, "Val   y:", y_val.shape)
print("Test  X:", X_test.shape, "Test  y:", y_test.shape)


Train X: (30530, 73) Train y: (30530,)
Val   X: (13482, 73) Val   y: (13482,)
Test  X: (7065, 73) Test  y: (7065,)


In [5]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE only to the training set
sm = SMOTE(random_state=42, sampling_strategy="auto", k_neighbors=1)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:\n", y_train_res.value_counts())
print("Resampled X:", X_train_res.shape)


Before SMOTE: ballon_dor_winner
0    30519
1       11
Name: count, dtype: int64
After SMOTE:
 ballon_dor_winner
0    30519
1    30519
Name: count, dtype: int64
Resampled X: (61038, 73)
